In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate
Data="jla_mub_0.txt"
Covmat="jla_mub_covmatrix.txt"
def load_data(Data, Covmat):
 data=np.loadtxt(Data)
 z=data[:,0]
 mu=data[:,1]
 cov_flat=np.loadtxt(Covmat)
 n=len(z)
 C=cov_flat.reshape(n,n)
 Cinv     = np.linalg.inv(C)
 return z, mu, C, Cinv

In [2]:
clight=2.99792458e5
def eta(a,Om):
 s=((1.0-Om)/Om)**(1.0/3.0)
 return (2.0*np.sqrt(1.0/Om)*(1.0/a**4-0.1540*s/a**3+0.4304*s**2/a**2+0.19097*s**3/a+0.066941*s**4)**(-1.0/8.0))

In [3]:
def luminosity_distance_flat(z,h,Om):
    a=1.0/(1.0+z)
    return (clight/(100.0*h))*(1.0+z)*(eta(1.0,Om)-eta(a,Om))

In [4]:
def mu_theoretical_flat(z,h,Om):
    DL_star=luminosity_distance_flat(z,h=1.0,Om=Om)
    return 25.0-5.0*np.log10(h)+5.0*np.log10(DL_star)

In [5]:
def log_likelihood_flat(params,z,mu_obs,Cinv):
 h,Om =params
 if h<=0.0 or Om<=0.0 or Om>=1.0:
  return -np.inf
 mu_th=mu_theoretical_flat(z, h, Om)
 delta=mu_obs-mu_th
 return -0.5*np.dot(delta,Cinv@delta)

In [6]:
def mh(log_prob_fn,initial_params,proposal_std,n_steps,args=()):
 n_params=len(initial_params)
 chain=np.zeros((n_steps,n_params))
 log_probs=np.zeros(n_steps)
 n_accepted=0
 theta_current=np.array(initial_params,dtype=float)
 lp_current=log_prob_fn(theta_current,*args)
 for i in range(n_steps):
  theta_proposed=theta_current+np.random.normal(0,proposal_std,n_params)
  lp_proposed=log_prob_fn(theta_proposed,*args)
  log_alpha=lp_proposed-lp_current
  if np.log(np.random.uniform())<log_alpha:
   theta_current=theta_proposed
   lp_current=lp_proposed
   n_accepted+=1
  chain[i]=theta_current
  log_probs[i]=lp_current
 return chain,log_probs,n_accepted

In [7]:
z,mu,C,Cinv=load_data(Data,Covmat)
theta0_flat=np.array([0.70,0.30])

In [8]:
fig,axes=plt.subplots(2,2,figsize=(12,6))
fig.suptitle("Chain traces for small vs large proposal sigma")
for col,(label,sigma) in enumerate([("Small s=0.001", 0.001),("Large s=1.0",1.0)]):
 chain_a,_,n_acc_a=mh(log_likelihood_flat,theta0_flat,sigma,n_steps=1000,args=(z,mu,Cinv))
 acc=n_acc_a/1000
 print(f"{label}: acceptance rate = {acc:.2%}")
 axes[0, col].plot(chain_a[:, 0], lw=0.8)
 axes[0, col].set_title(f"{label}  (accept={acc:.0%})")
 axes[0, col].set_ylabel("h")
 axes[1, col].plot(chain_a[:, 1], lw=0.8)
 axes[1, col].set_ylabel("Om")
 axes[1, col].set_xlabel("Steps")
plt.tight_layout()
plt.savefig("task1.png", dpi=120)
plt.show()

Small s=0.001: acceptance rate = 95.00%
Large s=1.0: acceptance rate = 0.00%


In [9]:
proposal_sizes=np.logspace(-3,2,30)
acc_rates=[]
for sigma in proposal_sizes:
 _,_, n_acc_b = mh(log_likelihood_flat, theta0_flat, sigma,n_steps=1000,args=(z,mu,Cinv))
 acc_rates.append(n_acc_b / 1000)
acc_rates = np.array(acc_rates)

In [10]:
plt.figure(figsize=(10, 8))
plt.semilogx(proposal_sizes,acc_rates,'o-')
plt.axhline(0.25,color='green', linestyle='--', label='Target ~25%')
plt.xlabel("Proposal sigma"); plt.ylabel("Acceptance rate")
plt.title("Acceptance rate vs proposal size")
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig("task2.png", dpi=120)
plt.show()

In [11]:
best_idx=np.argmin(np.abs(acc_rates-0.25))
best_sigma=proposal_sizes[best_idx]
print(f"Chosen sigma={best_sigma:.4f} (acceptance = {acc_rates[best_idx]:.2%})")

Chosen sigma=0.0240 (acceptance = 24.20%)


In [12]:
N_STEPS  = 50000
N_BURNIN = 5000

In [13]:
chain_full,lp_full,n_acc_c=mh(log_likelihood_flat,theta0_flat,best_sigma,n_steps=N_STEPS,args=(z, mu, Cinv))
print(f"Full-chain acceptance rate = {n_acc_c / N_STEPS:.2%}")

Full-chain acceptance rate = 24.45%


In [14]:
chain_flat  = chain_full[N_BURNIN:]
h_samples   = chain_flat[:,0]
Om_samples  = chain_flat[:,1]

In [15]:
h_mean=h_samples.mean()
Om_mean=Om_samples.mean()
h_std=h_samples.std()
Om_std=Om_samples.std()
cov_mat =np.cov(h_samples,Om_samples)

In [16]:
print(f"\n--- Flat-Universe MCMC Posterior ({len(chain_flat)} post-burn samples) ---")
print(f"  h  = {h_mean:.4f} +/- {h_std:.4f}")
print(f"  Om = {Om_mean:.4f} +/- {Om_std:.4f}")
print(f"  Cov(h, Om)  = {cov_mat[0, 1]:.6f}")
print(f"  Corr(h, Om) = {cov_mat[0, 1] / (h_std * Om_std):.4f}")


--- Flat-Universe MCMC Posterior (45000 post-burn samples) ---
  h  = 0.7023 +/- 0.0071
  Om = 0.2970 +/- 0.0314
  Cov(h, Om)  = -0.000137
  Corr(h, Om) = -0.6148


In [17]:
print("\n" + "="*60)
print("EXTENSION: Importance Sampling with Gaussian prior on h")
print("="*60)


EXTENSION: Importance Sampling with Gaussian prior on h


In [18]:
h_prior_mean  = 0.738
h_prior_sigma = 0.024

In [19]:
log_weights = -0.5 * ((h_samples - h_prior_mean) / h_prior_sigma) ** 2
log_weights -= log_weights.max()
weights = np.exp(log_weights)
weights /= weights.sum()       
h_IS  = np.sum(weights * h_samples)
Om_IS = np.sum(weights * Om_samples)
ESS = 1.0 / np.sum(weights ** 2)
print(f"Effective sample size (ESS) = {ESS:.0f}  (out of {len(weights)})")
print(f"\n  Without prior:  h = {h_mean:.4f},  Om = {Om_mean:.4f}")
print(f"  With IS prior:  h = {h_IS:.4f},  Om = {Om_IS:.4f}")
print(f"  (prior pulls h toward {h_prior_mean})")

Effective sample size (ESS) = 38518  (out of 45000)

  Without prior:  h = 0.7023,  Om = 0.2970
  With IS prior:  h = 0.7052,  Om = 0.2892
  (prior pulls h toward 0.738)


In [20]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Importance Sampling: Gaussian prior on h")
axes[0].hist(h_samples, bins=60, density=True, alpha=0.5, label="Uniform prior")
wh_counts, wh_edges = np.histogram(h_samples, bins=60,weights=weights,density=True)
axes[0].stairs(wh_counts, wh_edges, color='red', label="After Gaussian prior", lw=2)
axes[0].axvline(h_IS,   color='red',   linestyle='--', label=f"IS mean = {h_IS:.3f}")
axes[0].axvline(h_mean, color='navy',  linestyle='--', label=f"Unw. mean = {h_mean:.3f}")
h_grid = np.linspace(h_samples.min(), h_samples.max(), 200)
prior_curve = np.exp(-0.5 * ((h_grid - h_prior_mean) / h_prior_sigma)**2)
prior_curve /= prior_curve.max() * (wh_counts.max() / prior_curve.max() + 1e-10)
axes[0].plot(h_grid, prior_curve / prior_curve.max() * wh_counts.max(),'k:', lw=1.5, label="Prior shape")
axes[0].set_xlabel("h"); axes[0].set_ylabel("Density")
axes[0].legend(fontsize=8)
axes[1].hist(Om_samples, bins=60, density=True, alpha=0.5, label="Uniform prior")
wh_counts2, wh_edges2 = np.histogram(Om_samples, bins=60,weights=weights, density=True)
axes[1].stairs(wh_counts2, wh_edges2, color='red', label="After Gaussian prior", lw=2)
axes[1].axvline(Om_IS,   color='red',  linestyle='--', label=f"IS mean = {Om_IS:.3f}")
axes[1].axvline(Om_mean, color='navy', linestyle='--', label=f"Unw. mean = {Om_mean:.3f}")
axes[1].set_xlabel(r"$\Omega_m$"); axes[1].set_ylabel("Density")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("importance_sampling.png", dpi=120)
plt.show()

In [21]:
def gelman_rubin(chains):
 M=len(chains)
 n=chains[0].shape[0]
 n_params=chains[0].shape[1]
 chain_arr=np.stack(chains,axis=0)
 chain_means=chain_arr.mean(axis=1)
 chain_vars=chain_arr.var(axis=1, ddof=1)
 grand_mean=chain_means.mean(axis=0)
 B=n/(M-1)*np.sum((chain_means-grand_mean)**2,axis=0)
 W=chain_vars.mean(axis=0)
 Var_hat=(1.0-1.0/n)*W+B/n
 R_hat=np.sqrt(Var_hat/W)
 return R_hat

In [22]:
print("\n" + "="*60)
print("EXTENSION: Gelman-Rubin Convergence Test")
print("="*60)


EXTENSION: Gelman-Rubin Convergence Test


In [23]:
N_GR_CHAINS=4
N_GR_STEPS=20000
N_GR_BURNIN=2000
gr_starts=[[0.65, 0.25], [0.70, 0.30], [0.75, 0.35], [0.68, 0.28]]
gr_chains=[]

In [24]:
for k, start in enumerate(gr_starts):
 ch, _, _ = mh(log_likelihood_flat, np.array(start), best_sigma,n_steps=N_GR_STEPS, args=(z, mu, Cinv))
 gr_chains.append(ch[N_GR_BURNIN:])
 print(f"  Chain {k+1} done  (start={start})")

  Chain 1 done  (start=[0.65, 0.25])
  Chain 2 done  (start=[0.7, 0.3])
  Chain 3 done  (start=[0.75, 0.35])
  Chain 4 done  (start=[0.68, 0.28])


In [25]:
lengths   = np.linspace(100, len(gr_chains[0]), 30, dtype=int)
rhat_over_time = []   # will be shape (n_lengths, n_params)

In [26]:
for L in lengths:
 sub=[c[:L] for c in gr_chains]
 rhat_over_time.append(gelman_rubin(sub))

In [27]:
rhat_over_time = np.array(rhat_over_time)

In [28]:
R_hat_final = gelman_rubin(gr_chains)
print(f"\nFinal Gelman-Rubin R-hat (using {len(gr_chains[0])} post-burn samples per chain):")
print(f"  R_hat(h)  = {R_hat_final[0]:.4f}")
print(f"  R_hat(Om) = {R_hat_final[1]:.4f}")
print("  (values < 1.1 indicate convergence)")


Final Gelman-Rubin R-hat (using 18000 post-burn samples per chain):
  R_hat(h)  = 1.0010
  R_hat(Om) = 1.0021
  (values < 1.1 indicate convergence)


In [29]:
for param_idx, name in enumerate(["h", "Om"]):
 below = np.where(rhat_over_time[:, param_idx] < 1.1)[0]
 if len(below) > 0:
  needed = lengths[below[0]] + N_GR_BURNIN   # add back burn-in
  print(f"  R_hat({name}) first drops below 1.1 at ~{needed} total steps")
 else:
  print(f"  R_hat({name}) never fell below 1.1 — chain needs to be longer")

  R_hat(h) first drops below 1.1 at ~2100 total steps
  R_hat(Om) first drops below 1.1 at ~2100 total steps


In [30]:
plt.figure(figsize=(8, 4))
plt.plot(lengths, rhat_over_time[:, 0], label=r"$\hat{R}(h)$")
plt.plot(lengths, rhat_over_time[:, 1], label=r"$\hat{R}(\Omega_m)$")
plt.axhline(1.1, color='red', linestyle='--', label="Threshold 1.1")
plt.xlabel("Post-burn-in samples per chain")
plt.ylabel(r"$\hat{R}$")
plt.title(f"Gelman-Rubin convergence ({N_GR_CHAINS} chains)")
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig("gelman_rubin.png", dpi=120)
plt.show()

In [31]:
def numerical_gradient(log_prob_fn, theta, args=(), eps=1e-5):
 grad = np.zeros_like(theta)
 for i in range(len(theta)):
  theta_p = theta.copy(); theta_p[i] += eps
  theta_m = theta.copy(); theta_m[i] -= eps
  grad[i] = (log_prob_fn(theta_p, *args) - log_prob_fn(theta_m, *args)) / (2.0 * eps)
 return -grad

In [32]:
def gaussian_gradient(theta, theta_mean, Cinv_post):
 return Cinv_post @ (theta - theta_mean)

In [33]:
def leapfrog(theta, u, grad_U_fn, epsilon, n_steps):
 theta=theta.copy()
 u=u.copy()
 u-= 0.5 * epsilon * grad_U_fn(theta)
 for _ in range(n_steps - 1):
  theta += epsilon * u
  u     -= epsilon * grad_U_fn(theta)
 theta += epsilon * u
 u     -= 0.5 * epsilon * grad_U_fn(theta)
 return theta, u

In [34]:
def hmc(log_prob_fn, grad_U_fn, initial_params, epsilon, n_leapfrog,n_steps, args=()):
 n_params=len(initial_params)
 chain=np.zeros((n_steps, n_params))
 log_probs=np.zeros(n_steps)
 n_accepted=0
 theta_current=np.array(initial_params, dtype=float)
 lp_current=log_prob_fn(theta_current, *args)
 for i in range(n_steps):
  u_current = np.random.normal(0.0, 1.0, n_params)
  theta_proposed, u_proposed = leapfrog(theta_current, u_current, grad_U_fn, epsilon, n_leapfrog)
  lp_proposed = log_prob_fn(theta_proposed, *args)
  H_current  = -lp_current  + 0.5 * np.dot(u_current,  u_current)
  H_proposed = -lp_proposed + 0.5 * np.dot(u_proposed, u_proposed)
  delta_H = H_proposed - H_current
  if np.log(np.random.uniform()) < -delta_H:
   theta_current=theta_proposed
   lp_current=lp_proposed
   n_accepted+=1
  chain[i]=theta_current
  log_probs[i]=lp_current
 return chain,log_probs,n_accepted

In [35]:
print("\n" + "="*60)
print("EXTENSION: Hamiltonian Monte Carlo (HMC)")
print("="*60)


EXTENSION: Hamiltonian Monte Carlo (HMC)


In [36]:
theta_mcmc_mean = np.array([h_mean, Om_mean])
C_post= cov_mat
C_post_inv=np.linalg.inv(C_post)

In [37]:
grad_U_gaussian = lambda theta: gaussian_gradient(theta, theta_mcmc_mean, C_post_inv)
grad_U_numerical = lambda theta: numerical_gradient(log_likelihood_flat, theta, args=(z, mu, Cinv))
epsilon_hmc = 0.005 
n_leapfrog_hmc = 20
N_HMC = 10000
N_HMC_BURNIN = 500

In [38]:
chain_hmc_A,lp_hmc_A,n_acc_A=hmc(log_likelihood_flat,grad_U_gaussian,theta_mcmc_mean,epsilon_hmc,n_leapfrog_hmc,n_steps=N_HMC, args=(z, mu, Cinv))
acc_A = n_acc_A/N_HMC
print(f"HMC (Gaussian grad) acceptance rate = {acc_A:.2%}")

HMC (Gaussian grad) acceptance rate = 95.93%


In [39]:
chain_hmc_B,lp_hmc_B,n_acc_B=hmc(log_likelihood_flat,grad_U_numerical,theta_mcmc_mean,epsilon_hmc,n_leapfrog_hmc,n_steps=N_HMC, args=(z, mu, Cinv))
acc_B = n_acc_B / N_HMC
print(f"HMC (numerical grad) acceptance rate = {acc_B:.2%}")

HMC (numerical grad) acceptance rate = 98.64%


In [40]:
hmc_A_post = chain_hmc_A[N_HMC_BURNIN:]
hmc_B_post = chain_hmc_B[N_HMC_BURNIN:]

In [41]:
print(f"\nHMC posteriors ({len(hmc_A_post)} post-burn samples):")
print(f"  [Gaussian grad]  h = {hmc_A_post[:,0].mean():.4f}  Om = {hmc_A_post[:,1].mean():.4f}")
print(f"  [Numerical grad] h = {hmc_B_post[:,0].mean():.4f}  Om = {hmc_B_post[:,1].mean():.4f}")


HMC posteriors (9500 post-burn samples):
  [Gaussian grad]  h = 0.7018  Om = 0.2961
  [Numerical grad] h = 0.7021  Om = 0.2974


In [42]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("HMC vs Metropolis-Hastings: chain traces")
for row, (name, samples) in enumerate([("MH (flat chain)",chain_flat),("HMC numerical grad",hmc_B_post)]):
 axes[row, 0].plot(samples[:2000, 0], lw=0.6, color='tab:blue')
 axes[row, 0].set_title(f"{name} — h")
 axes[row, 0].set_ylabel("h")
 axes[row, 1].plot(samples[:2000, 1], lw=0.6, color='tab:orange')
 axes[row, 1].set_title(f"{name} — Om")
 axes[row, 1].set_ylabel(r"$\Omega_m$")
for ax in axes[1]:
 ax.set_xlabel("Sample index")
plt.tight_layout()
plt.savefig("hmc_vs_mh_traces.png", dpi=120)
plt.show()

In [43]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Parameter space exploration: HMC vs MH")
N_show = 3000
axes[0].scatter(chain_flat[:N_show, 0], chain_flat[:N_show, 1],s=1, alpha=0.3, color='tab:blue', label="MH")
axes[0].set_title("Metropolis-Hastings")
axes[0].set_xlabel("h"); axes[0].set_ylabel(r"$\Omega_m$")
axes[1].scatter(hmc_B_post[:N_show, 0], hmc_B_post[:N_show, 1],s=1, alpha=0.3, color='tab:red', label="HMC")
axes[1].set_title("HMC (numerical gradient)")
axes[1].set_xlabel("h"); axes[1].set_ylabel(r"$\Omega_m$")
plt.tight_layout()
plt.savefig("hmc_vs_mh_scatter.png", dpi=120)
plt.show()

In [44]:
def autocorrelation(x, max_lag=200):
 x=x-x.mean()
 var=np.var(x)
 ac=np.array([np.mean(x[:len(x)-k] * x[k:]) / var for k in range(max_lag)])
 return ac

In [45]:
lags = np.arange(200)
ac_mh_h   = autocorrelation(chain_flat[:, 0])
ac_hmc_h  = autocorrelation(hmc_B_post[:, 0])
ac_mh_Om  = autocorrelation(chain_flat[:, 1])
ac_hmc_Om = autocorrelation(hmc_B_post[:, 1])

In [46]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Autocorrelation: HMC vs MH  (lower = more efficient)")
axes[0].plot(lags, ac_mh_h,  label="MH")
axes[0].plot(lags, ac_hmc_h, label="HMC")
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_title("Autocorrelation of h")
axes[0].set_xlabel("Lag"); axes[0].set_ylabel("ACF")
axes[0].legend()
axes[1].plot(lags, ac_mh_Om,  label="MH")
axes[1].plot(lags, ac_hmc_Om, label="HMC")
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_title(r"Autocorrelation of $\Omega_m$")
axes[1].set_xlabel("Lag"); axes[1].set_ylabel("ACF")
axes[1].legend()
plt.tight_layout()
plt.savefig("autocorrelation_hmc_mh.png", dpi=120)
plt.show()

In [47]:
print("\n" + "="*60)
print("SUMMARY OF ALL POSTERIOR ESTIMATES")
print("="*60)
print(f"{'Method':<30} {'h':>10} {'Om':>10}")
print("-"*52)
print(f"{'MH (flat, uniform prior)':<30} {h_mean:>10.4f} {Om_mean:>10.4f}")
print(f"{'MH (flat, IS Gaussian prior)':<30} {h_IS:>10.4f} {Om_IS:>10.4f}")
print(f"{'HMC Gaussian grad':<30} {hmc_A_post[:,0].mean():>10.4f} {hmc_A_post[:,1].mean():>10.4f}")
print(f"{'HMC numerical grad':<30} {hmc_B_post[:,0].mean():>10.4f} {hmc_B_post[:,1].mean():>10.4f}")


SUMMARY OF ALL POSTERIOR ESTIMATES
Method                                  h         Om
----------------------------------------------------
MH (flat, uniform prior)           0.7023     0.2970
MH (flat, IS Gaussian prior)       0.7052     0.2892
HMC Gaussian grad                  0.7018     0.2961
HMC numerical grad                 0.7021     0.2974
